In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, DATA_DIR as METAB_DATA_DIR

XENIUM_DATA_DIR = PROJECT_DATA_DIR
DATASET_NAME = 'Primary_Dermal_Melanoma'
DATA_DIR = f'{XENIUM_DATA_DIR}/{DATASET_NAME}'

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad(f'{DATA_DIR}/adata.h5ad')
adata

In [ ]:
sc.pl.spatial(adata, spot_size=5, show=False, title=DATASET_NAME, color='Tier3');

In [ ]:
sc.pl.spatial(
    adata[adata.obs['Tier3'] != 'other'], 
    spot_size=35, 
    show=False, 
    title=DATASET_NAME, 
    color='Tier3'
);

In [ ]:
from metab_processing.SpaceTravLR import beta_analysis

betadata_dir = f'{DATA_DIR}/spacetravlr_output/betadata'
beta_analysis.betas_to_adata(adata, betadata_dir, group=None)
adata

In [ ]:
list(adata.uns['beta_modulators'])

In [ ]:
GENE = 'CD4'
adata.uns['beta_modulators'][GENE]

In [ ]:
def plot_beta(adata, gene, modulator, tier, cell_types=None, spot_size=35):
    i = adata.uns['beta_modulators'][gene].index(modulator)
    sub = adata if cell_types is None else adata[adata.obs[tier].isin(cell_types)]
    sub = sub[:, :1].copy()
    sub.obs['beta'] = sub.obsm[f'beta_{gene}'][:, i]
    sc.pl.spatial(sub, color='beta', spot_size=spot_size, show=False,
                  title=f'{gene} ~ {modulator} ({tier})')

In [ ]:
gp = [m for m in adata.uns['beta_modulators'][GENE] if '@' in m][0]
plot_beta(adata, GENE, gp, 'Tier1', ['T Cell'])